# Preparing the Habermas Machine dataset for the Fig. 4C replication

The public release accompanying Tessler et al. (2024) is three parquet files in `data/`:

* `hm_all_candidate_comparisons.parquet` — one row per participant per round per iteration: the
  participant's own free-text opinion, the group statement they critiqued, the full candidate list
  the mediator produced at that iteration, their ratings and rankings.
* `hm_all_position_statement_ratings.parquet` — each participant's Likert agreement with the
  question's position statement, before (`rating_index` 0) and after (`rating_index` 1) deliberation.
* `hm_all_round_survey_responses.parquet` — the end-of-round survey, which is where the *revised*
  group statement is found.

This notebook reshapes them into the five tables the analysis reads, and writes them to `prepared/`:

| table | one row per | key columns |
| --- | --- | --- |
| `opinions.parquet` | participant-round | opinion text, pre- and post-deliberation Likert rating |
| `statements.parquet` | round | the initial group statement and the revised group statement |
| `candidates.parquet` | candidate statement | every candidate the mediator produced, and which one won |
| `questions.parquet` | question | question text and its affirming / negating position statements |
| `texts.parquet` | unique text | every text that has to be embedded, with an embedding priority |

Every table is keyed on `metadata.version` (the cohort), `launch_id` (the group) and `round_id`.
Run `python reproduce.py data` first if `data/` is empty.

In [1]:
import sys
sys.path.insert(0, "..")

from pathlib import Path

import pandas as pd

from hm_fig4c import data as D
from hm_fig4c.preprocess import preregistered_launches

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

DATA_DIR, OUT_DIR = Path("../data"), Path("../prepared")


## 1. The raw comparisons table

`load_comparisons` reads the columns the analysis needs. One row is one participant's view of one
iteration of one round, so the same round appears once per participant per iteration.

In [2]:
comps = D.load_comparisons(DATA_DIR)
print("rows:", comps.shape)
comps[["metadata.version", "launch_id", "round_id", "iteration_index", "metadata.participant_id",
       "question.id", "own_opinion.text"]].head()

rows: (39955, 28)


,metadata.version,launch_id,round_id,iteration_index,metadata.participant_id,question.id,own_opinion.text
0,TRAINING_DATA_V1,17b6daea928a41c6a5727aa0c5d674b9,315be495e0403a1570c0d3ed7ae548b9,0,35a875f9b5,S173961439,I dont think parents should be able to opt out...
1,TRAINING_DATA_V1,17b6daea928a41c6a5727aa0c5d674b9,315be495e0403a1570c0d3ed7ae548b9,0,196beeb338,S173961439,I don't believe they should be allowed to opt ...
2,TRAINING_DATA_V1,17b6daea928a41c6a5727aa0c5d674b9,315be495e0403a1570c0d3ed7ae548b9,0,6036bc29ed,S173961439,I think sex education should be somewhat manda...
3,TRAINING_DATA_V1,17b6daea928a41c6a5727aa0c5d674b9,315be495e0403a1570c0d3ed7ae548b9,0,49d58e7851,S173961439,In my opinion I feel like every child should l...
4,TRAINING_DATA_V1,17b6daea928a41c6a5727aa0c5d674b9,315be495e0403a1570c0d3ed7ae548b9,0,b650c3ef1c,S173961439,"On the whole, I don't they they should. It is ..."


The release covers the training runs, the four evaluation cohorts and the Virtual Citizens'
Assembly. This report uses the main-task ablation cohorts 1--3.

In [3]:
comps["metadata.version"].value_counts().rename("rows").to_frame()

,rows
metadata.version,
TRAINING_DATA_V1,6955
EVAL_COHORT2_ABLATION_IID_V2,5920
EVAL_COHORT1_ABLATION_IID_V1,5085
EVAL_COHORT3_ABLATION_OOD_V1,3920
TRAINING_DATA_V2,3845
EVAL_COHORT4_CRITIQUE_EXCLUSION,2790
TRAINING_DATA_V3,2555
EVAL_COHORT6_HUMAN_MEDIATOR,2150
TRAINING_DATA_V4,1580


## 2. Build the prepared tables

`hm_fig4c.data.prepare` calls `build_tables` (which reshapes the three raw files and asserts that
the initial and revised statements are unique within a round) and `build_text_table`, then writes
the five parquet files into `prepared/`. Its printout is the build's own sanity check.

In [4]:
opinions, statements, questions, candidates, texts = D.prepare(DATA_DIR, OUT_DIR)

opinions: (16243, 12) | pre-rating coverage: 0.9968
cohort
cohort1     2262
cohort2     2607
cohort3     1729
cohort4     1211
training    7164
vca         1270
dtype: int64
statements: (3782, 12)
pre-registered groups / rounds per version:
                                  groups  rounds
metadata.version                               
EVAL_COHORT1_ABLATION_IID_V1        100     300
EVAL_COHORT2_ABLATION_IID_V2        150     450
EVAL_COHORT3_ABLATION_OOD_V1         99     297
EVAL_COHORT4_CRITIQUE_EXCLUSION      50     150
          initial_id  revised_id
cohort                          
cohort1          1.0       0.998
cohort2          1.0       1.000
cohort3          1.0       0.997
cohort4          1.0       1.000
training         1.0       1.000
vca              1.0       1.000
questions: (2250, 6)
candidates: (29889, 10) | per round-iteration: {4: 7359, 2: 148, 3: 50, 1: 7}
winner found among candidates: 0.9997


texts to embed: 53833
priority  kind                       
-2        position_generic                  2
          position_prefixed              4495
          position_prefixed_not_lower    2069
-1        position                       4493
 0        initial                         506
          opinion                        2262
          revised                         433
 1        initial                         586
          opinion                        2607
          revised                         495
 2        initial                         391
          opinion                        1729
          revised                         337
 3        candidate                      5388
          initial                         276
          opinion                        1211
          revised                         247
 4        initial                        1662
          opinion                        7164
          revised                        1343
 5        initial   

### `opinions` — one row per participant-round

`pre_rating` is the 1--7 Likert rating given before deliberation; it is what defines the minority
side of a group.

In [5]:
print("opinions:", opinions.shape)
opinions.head()

opinions: (16243, 12)


,metadata.version,launch_id,round_id,cohort,prereg,participant_id,worker_id,question_id,opinion_id,opinion_text,pre_rating,post_rating
0,TRAINING_DATA_V1,17b6daea928a41c6a5727aa0c5d674b9,315be495e0403a1570c0d3ed7ae548b9,training,False,35a875f9b5,NaN,S173961439,5d65e7df5d7be2b93ea42748b54209c7,I dont think parents should be able to opt out...,2.0,2.0
1,TRAINING_DATA_V1,17b6daea928a41c6a5727aa0c5d674b9,315be495e0403a1570c0d3ed7ae548b9,training,False,196beeb338,NaN,S173961439,7a5c677cccfb0c2e5539388c05f1bef0,I don't believe they should be allowed to opt ...,2.0,6.0
2,TRAINING_DATA_V1,17b6daea928a41c6a5727aa0c5d674b9,315be495e0403a1570c0d3ed7ae548b9,training,False,6036bc29ed,NaN,S173961439,7f2605ec99d04eb759e7f59cfcd47771,I think sex education should be somewhat manda...,2.0,1.0
3,TRAINING_DATA_V1,17b6daea928a41c6a5727aa0c5d674b9,315be495e0403a1570c0d3ed7ae548b9,training,False,49d58e7851,NaN,S173961439,1849f11739c7ff1fce7cccb7c9765446,In my opinion I feel like every child should l...,2.0,1.0
4,TRAINING_DATA_V1,17b6daea928a41c6a5727aa0c5d674b9,315be495e0403a1570c0d3ed7ae548b9,training,False,b650c3ef1c,NaN,S173961439,f5ed054321f5a23a713a1cfd89116206,"On the whole, I don't they they should. It is ...",2.0,2.0


### `statements` — one row per round, with the two group statements

In [6]:
print("statements:", statements.shape)
statements[["metadata.version", "launch_id", "round_id", "question_id", "cohort", "prereg",
            "initial_text", "revised_text"]].head()

statements: (3782, 12)


,metadata.version,launch_id,round_id,question_id,cohort,prereg,initial_text,revised_text
0,EVAL_COHORT1_ABLATION_IID_V1,00295a73bd9147c28322021c19c72a30,49e3bcc23d457120cda370205f2df7b6,S173968782,cohort1,True,People in the UK should be more health conscio...,People in the UK should be more health conscio...
1,EVAL_COHORT1_ABLATION_IID_V1,00295a73bd9147c28322021c19c72a30,6d6b0a8beed0c713ad35bf1c827a7f20,S173964173,cohort1,True,While it is important to protect people on bus...,While it is important to protect people on bus...
2,EVAL_COHORT1_ABLATION_IID_V1,00295a73bd9147c28322021c19c72a30,abe5ca704bcae0c68d6f547c8c0f86fc,S173969028,cohort1,True,Employers should be forced to provide their em...,Final consensus statement: Employers should be...
3,EVAL_COHORT1_ABLATION_IID_V1,00f0fddde86a4c138aee07688086b7ad,63ad01f50671c50157763c57a76a6b0f,S173965133,cohort1,True,"No wild animal should be kept as a pet, even i...","No wild animal should be kept as a pet, even i..."
4,EVAL_COHORT1_ABLATION_IID_V1,00f0fddde86a4c138aee07688086b7ad,6c3abd065e6af21aa8ba2e388bf6da35,S173963520,cohort1,True,The development of AI has come on so quickly a...,The development of AI has come on so quickly a...


### `questions` — one row per question, with the two position statements that anchor the axis

In [7]:
print("questions:", questions.shape)
questions.head()

questions: (2250, 6)


,question_id,question_text,affirming,negating,topic,split
0,S173961439,Should parents be allowed to opt out of sex ed...,Parents should be allowed to opt out of sex ed...,Parents should NOT be allowed to opt out of se...,66,TRAIN
10,S173969345,Should the UK and France share the Channel Tun...,The UK and France should share the Channel Tun...,The UK and France should NOT share the Channel...,78,TRAIN
15,S173967025,Should we reintroduce wolves into the Scottish...,We should reintroduce wolves into the Scottish...,We should NOT reintroduce wolves into the Scot...,63,TRAIN
25,S173961579,Is it okay to use genetic engineering to give ...,It is okay to use genetic engineering to give ...,It is NOT okay to use genetic engineering to g...,43,TRAIN
40,S173966760,Is it better to give everyone a basic income o...,It is better to give everyone a basic income.,It is better to give everyone a a certain amou...,6,TRAIN


### `candidates` — every candidate statement the mediator produced

`phase` is `initial` (iteration 0) or `revised` (iteration 1); `is_winner` marks the candidate that
was selected by the simulated vote.

In [8]:
print("candidates:", candidates.shape)
candidates.head()

candidates: (29889, 10)


,metadata.version,launch_id,round_id,iteration_index,candidate_id,candidate_text,cohort,prereg,phase,is_winner
0,TRAINING_DATA_V1,17b6daea928a41c6a5727aa0c5d674b9,315be495e0403a1570c0d3ed7ae548b9,0,0807826b74f4425293b8f983c98f3e06,"In general, the citizen's jury felt that the c...",training,False,initial,False
1,TRAINING_DATA_V1,17b6daea928a41c6a5727aa0c5d674b9,315be495e0403a1570c0d3ed7ae548b9,0,9ef1b8c9dfc245b4e54df929831bb3c2,Citizen's Jury,training,False,initial,False
2,TRAINING_DATA_V1,17b6daea928a41c6a5727aa0c5d674b9,315be495e0403a1570c0d3ed7ae548b9,0,364ddcc06751a3be8545c907418eb476,Children should receive sex education in schoo...,training,False,initial,False
3,TRAINING_DATA_V1,17b6daea928a41c6a5727aa0c5d674b9,315be495e0403a1570c0d3ed7ae548b9,0,b1e0a2ad180aa663489c50854487f600,"Sex education is very important in schools, as...",training,False,initial,True
4,TRAINING_DATA_V1,17b6daea928a41c6a5727aa0c5d674b9,315be495e0403a1570c0d3ed7ae548b9,1,182b633d6567f190a0da6dcaff9929b3,"Sex education is very important in schools, as...",training,False,revised,False


## 3. The pre-registration filter

`preregistered_launches` ports the pre-registered preprocessing from the authors' released
`live_loading.py`: drop groups containing repeat participants, then keep the first *N* groups that
reach the pre-registered minimum size. `build_tables` records the result as the `prereg` column.

In [9]:
prereg = preregistered_launches(comps)
{v: len(ids) for v, ids in prereg.items()}

{'EVAL_COHORT1_ABLATION_IID_V1': 100,
 'EVAL_COHORT2_ABLATION_IID_V2': 150,
 'EVAL_COHORT3_ABLATION_OOD_V1': 99,
 'EVAL_COHORT4_CRITIQUE_EXCLUSION': 50}

In [10]:
def counts(df):
    return df.groupby("metadata.version").agg(groups=("launch_id", "nunique"), rounds=("round_id", "size"))

main = statements[statements["metadata.version"].isin(D.COHORTS["cohorts_1_3"])]
counts(main).join(counts(main[main["prereg"]]), lsuffix="_all", rsuffix="_prereg")

,groups_all,rounds_all,groups_prereg,rounds_prereg
metadata.version,,,,
EVAL_COHORT1_ABLATION_IID_V1,176,508,100,300
EVAL_COHORT2_ABLATION_IID_V2,198,592,150,450
EVAL_COHORT3_ABLATION_OOD_V1,132,392,99,297


## 4. The texts to embed

One row per unique text. `priority` orders the embedding run so that the texts this report needs
(the position-statement endpoints and everything from the pre-registered rounds of cohorts 1--3)
come first; `hm_fig4c.embed.run` embeds priority 3 and below.

In [11]:
print("texts:", texts.shape)
texts.head()

texts: (53833, 6)


,text_id,text,kind,priority,prereg,n_words
0,3e0740c3a1bf4f91,"Yes, I agree. The government should impose a c...",position_prefixed,-2,True,10
1,c6e5ed2afc022b1a,"No, I disagree. The government should NOT impo...",position_prefixed,-2,True,11
2,fdc72ea21a335716,"No, I disagree. The government should not impo...",position_prefixed_not_lower,-2,True,11
3,331eab1441eed380,"Yes, I agree. There should be a ban on small p...",position_prefixed,-2,True,13
4,b9f5c06c72ce72bc,"No, I disagree. There should NOT be a ban on s...",position_prefixed,-2,True,14


In [12]:
texts.groupby("kind")["n_words"].describe(percentiles=[.5, .9, .99]).round(1)

,count,mean,std,min,50%,90%,99%,max
kind,,,,,,,,
candidate,19658.0,100.1,54.4,0.0,93.0,181.0,227.0,244.0
initial,3737.0,112.1,45.4,10.0,107.0,176.0,223.0,240.0
opinion,16243.0,63.3,29.8,11.0,59.0,103.0,151.6,199.0
position,4493.0,11.3,4.0,2.0,11.0,16.0,24.0,31.0
position_generic,2.0,3.0,0.0,3.0,3.0,3.0,3.0,3.0
position_prefixed,4495.0,14.3,4.0,5.0,14.0,19.0,27.0,34.0
position_prefixed_not_lower,2069.0,14.9,3.9,7.0,14.0,20.0,28.0,34.0
revised,3136.0,136.4,49.2,9.0,137.0,205.0,227.7,239.0


## 5. What was written

These five files are the input to `notebooks/analysis.ipynb`; `texts.parquet` is also the input to
`scripts/embed.py`.

In [13]:
pd.DataFrame([{"file": p.name, "MB": round(p.stat().st_size / 1e6, 2)}
              for p in sorted(OUT_DIR.glob("*.parquet"))])

,file,MB
0,candidates.parquet,7.19
1,opinions.parquet,4.19
2,questions.parquet,0.26
3,statements.parquet,3.12
4,texts.parquet,10.82
